# Stage 3: Token Matching — Demo

This notebook demonstrates all matching methods on a single sample from the SECOND dataset.

**Pipeline position**: Stage 1 (SAM2 embeddings) → Stage 2 (region tokenization) → **Stage 3 (token matching)** → Stage 4 (change reasoning)


In [ ]:
import sys, torch
sys.path.insert(0, '..')

from token_matching import MatchConfig, TokenMatcher
from token_matching_utils import (
    normalize_embeddings, fused_similarity_matrix,
    soft_matrix_softmax, soft_matrix_sinkhorn,
    plot_matches, plot_soft_heatmap,
)
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1. Load a sample token pair

In [ ]:
T1_DIR = Path('../SECOND/tokens_T1')
T2_DIR = Path('../SECOND/tokens_T2')

stems = sorted(p.stem for p in T1_DIR.glob('*.pt'))
stem  = stems[0]
print(f'Sample: {stem}')

data_t1 = torch.load(T1_DIR / f'{stem}.pt', weights_only=True)
data_t2 = torch.load(T2_DIR / f'{stem}.pt', weights_only=True)

print(f'T1 tokens: {data_t1["tokens"].shape}')
print(f'T2 tokens: {data_t2["tokens"].shape}')
print(f'T1 centroids: {data_t1["centroids"].shape}')
print(f'T1 areas :: min={data_t1["areas"].min():.4f}  max={data_t1["areas"].max():.4f}')

## 2. Cosine similarity matrix

In [ ]:
tok1 = normalize_embeddings(data_t1['tokens'].float())
tok2 = normalize_embeddings(data_t2['tokens'].float())
cen1 = data_t1['centroids'].float()
cen2 = data_t2['centroids'].float()

sim, cos_raw, stats = fused_similarity_matrix(
    tok1, tok2, cen1, cen2,
    alpha=1.0, beta=0.5, spatial_gate_dist=0.3
)
print('Fused similarity shape:', sim.shape)
print('Stats:', stats)

## 3. Compare all matching methods

In [ ]:
methods = ['cosine_spatial', 'nearest_neighbor', 'hungarian', 'soft', 'cross_attention']
results = {}

for method in methods:
    cfg = MatchConfig(
        method=method,
        device=DEVICE,
        spatial_gate_dist=0.3,
        top_k=10,
        visualize=False,
    )
    matcher = TokenMatcher(cfg)
    res = matcher.match(data_t1, data_t2)
    results[method] = res
    meta = res['metadata']
    print(f'{method:20s}  matches={meta["n_matches"]:4d}  "
          f"unmatched_T1={len(res["unmatched_T1"]):3d}  "
          f"unmatched_T2={len(res["unmatched_T2"]):3d}')

## 4. Visualize soft matching heatmap (Hungarian)

In [ ]:
from pathlib import Path
out = Path('/tmp/demo_match')
out.mkdir(exist_ok=True)

plot_soft_heatmap(stem, results['hungarian']['soft_matrix'], out_dir=out)
plot_matches(
    stem, cen1, cen2,
    pairs=results['hungarian']['pairs'],
    out_dir=out,
    areas_t1=data_t1['areas'], areas_t2=data_t2['areas'],
)

from IPython.display import Image, display
display(Image(filename=str(out / f'{stem}_soft_heatmap.png')))
display(Image(filename=str(out / f'{stem}_matches.png')))

## 5. Split / merge detection

In [ ]:
from token_matching_utils import detect_splits_merges

res = results['nearest_neighbor']
sm = detect_splits_merges(
    res['pairs'], data_t1['areas'], data_t2['areas'],
    area_ratio_threshold=0.6
)
print('Splits (T1 token indices):', sm['splits'])
print('Merges (T2 token indices):', sm['merges'])

## 6. Run on full dataset

```bash
python ../token_matching.py \
  --tokens_T1 ../SECOND/tokens_T1 \
  --tokens_T2 ../SECOND/tokens_T2 \
  --output ../SECOND/matches \
  --method hungarian \
  --device cuda \
  --visualize \
  --n_vis 20
```

Estimated time: ~2968 pairs × 42ms ≈ **~125 seconds** on a single GPU.
